# Push `part_type` to BDRC for existing segments

`part_type` is a field on BDRC's own volume/segment data, set through the `update_volume` (`POST /volumes/{volume_id}`) endpoint — it is **not** a column in this repo's local database, so no schema/model change is needed here. This notebook only reads the local outliner DB (for segment `label`, spans, reviewed title/author) and pushes the derived `part_type` to BDRC over the existing HTTP API.

Mapping from the local `label` field:

| `label` | `part_type` |
|---|---|
| `TEXT` | `text` |
| `FRONT_MATTER` / `BACK_MATTER` | `editorial` |
| `TOC` | `toc` |
| `None` (unlabeled) | omitted from the push (left as-is on BDRC) |

Approach: page through BDRC volumes via `bdrc.volume.get_volumes` in batches; for each volume, use its `id` to look up the matching local document by `OutlinerDocument.filename` (same match `outliner/controller/bdrc.py::assign_volume` uses); then build one `POST /volumes/{volume_id}` payload per document from its segments and send it.

Field extraction (`cstart`/`cend`/`title_bo`/`author_name_bo`/`mw_id`/`wa_id`) mirrors `outliner/controller/bdrc.py::_push_document_segments_to_bdrc`, the existing reference for how a segment sync to BDRC is built — only `part_type` is computed differently (from `label` instead of whether `wa_id` is set).

The POST is a **full replace** of a volume's segments on BDRC's side, same as the normal sync path, so this resends every segment field for a matched document, not just `part_type`. The volume's `status` is read back from BDRC (`get_volume`) and resent unchanged, so this does not move a volume through the review workflow.

`bdrc.volume.SegmentInput.part_type` is currently typed `Literal["text", "editorial"]` (no `"toc"`), and the task is to only touch this notebook, not backend code — so the POST body is built as a plain dict and sent directly with the shared `httpx` client instead of going through `SegmentInput`/`update_volume`, letting `"toc"` through. If `"toc"` should become a first-class value, `bdrc/volume.py`'s `SegmentInput.part_type` Literal needs updating separately.

**Do not run this yet.** `DRY_RUN` defaults to `True` below — it will only report what it *would* send, without calling BDRC. Flip it to `False` for a real run, after reviewing the dry-run output.

## 1. Connect to the local database

Read-only from this notebook's point of view — used only to look up documents/segments by BDRC volume id. Same bootstrap as `backend/outliner_cleaner/data_to_bdrc.ipynb`: targets `DATABASE_URL_PRODUCTION` from `backend/.env` (not the app's normal `DATABASE_URL`), patched into `core.config` before `core.database` builds its engine. Add `DATABASE_URL_PRODUCTION=...` to `backend/.env` if it isn't there yet.

In [ ]:
from pathlib import Path
import os
import sys


def _backend_root() -> Path:
    """Find the FastAPI backend folder (contains core/database.py)."""
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "core" / "database.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError(
        "Could not find backend root. Open this notebook from backend/ or "
        "backend/fix_part_type/ (or chdir there), then rerun."
    )


BACKEND_ROOT = _backend_root()
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from dotenv import load_dotenv

load_dotenv(BACKEND_ROOT / ".env", override=True)

prod_url = (os.getenv("DATABASE_URL_PRODUCTION") or "").strip()
if not prod_url:
    raise RuntimeError(
        "Set DATABASE_URL_PRODUCTION in backend/.env to the target PostgreSQL URL. "
        "This notebook does not use DATABASE_URL (local) so a local run can't be "
        "mistaken for a production one."
    )

# Import config so .env is loaded, then swap the URL before the engine is created.
import core.config as core_config

core_config.DATABASE_URL = prod_url

from core.database import SessionLocal

## 2. Imports and configuration

In [ ]:
import asyncio
from typing import Any, Dict, List, Optional

from bdrc.volume import (
    APPLICATION_JSON,
    BDRC_BACKEND_URL,
    get_http_client,
    get_volume,
    get_volumes,
    aclose_http_client,
)
from outliner.repository import outliner_repository as outliner_repo
from outliner.models.segment_enums import SegmentLabels

# --- Safety switch -----------------------------------------------------------
# True: only compute and print what would be sent, no HTTP POSTs to BDRC.
# False: actually POST each document's segments (with part_type) to BDRC.
DRY_RUN = True

# Pagination size for each get_volumes() call.
BATCH_SIZE = 50

# BDRC volume statuses to page through while discovering volumes. Only in_review and
# reviewed volumes are in scope for this backfill. The status actually re-sent in each
# push is read fresh per volume from get_volume() below, not from this list.
STATUSES: List[str] = ["in_review", "reviewed"]

# Restrict to a single BDRC batch_id, or leave empty to cover all batches.
BATCH_ID = ""

LABEL_TO_PART_TYPE: Dict[SegmentLabels, str] = {
    SegmentLabels.TEXT: "text",
    SegmentLabels.FRONT_MATTER: "editorial",
    SegmentLabels.BACK_MATTER: "editorial",
    SegmentLabels.TOC: "toc",
}

## 3. Page through BDRC volumes in batches

In [ ]:
async def iter_all_volumes():
    """Yield every volume dict across STATUSES, paging get_volumes() with offset/limit
    until a batch comes back empty. A failed status/page is logged and skipped rather
    than aborting the whole run (e.g. an invalid status value, a transient timeout)."""
    for status in STATUSES:
        offset = 0
        while True:
            try:
                page = await get_volumes(
                    status=status, batch_id=BATCH_ID, offset=offset, limit=BATCH_SIZE
                )
            except Exception as e:
                print(f"iter_all_volumes: failed status={status!r} offset={offset}: {e}")
                break
            items = page.get("items", [])
            if not items:
                break
            for item in items:
                yield item
            if len(items) < BATCH_SIZE:
                break
            offset += BATCH_SIZE

## 4. Build the BDRC segment payload for one document

Same field extraction as `_push_document_segments_to_bdrc` in `outliner/controller/bdrc.py` (`span_start`/`span_end` → `cstart`/`cend`, `reviewer_title`/`reviewer_author` falling back to `title`/`author`, `mw_id`/`wa_id`) — `part_type` is the one field computed differently here, from `label` rather than whether `wa_id` is set. `part_type` is omitted (not just set to `None`) for unlabeled segments so the push doesn't clobber whatever BDRC already has for them.

In [ ]:
def build_segment_payloads(document, volume: Dict[str, Any]):
    """Return (payloads_for_bdrc, audit_rows) for every segment of `document`.
    payloads_for_bdrc is the raw list to send as VolumeInput.segments (as plain dicts,
    not SegmentInput, so a computed part_type of "toc" isn't rejected by that model's
    current Literal["text", "editorial"])."""
    payloads = []
    audit_rows = []
    for segment in document.segments:
        part_type = LABEL_TO_PART_TYPE.get(segment.label)
        segment_title = segment.reviewer_title if segment.reviewer_title is not None else (segment.title or "")
        segment_author = segment.reviewer_author if segment.reviewer_author is not None else (segment.author or "")
        mw_id = f'{volume["mw_id"]}_{segment.id}'
        wa_id = segment.title_bdrc_id or ""

        payload = {
            "cstart": int(segment.span_start),
            "cend": int(segment.span_end),
            "title_bo": segment_title,
            "author_name_bo": segment_author,
            "mw_id": mw_id,
            "wa_id": wa_id,
        }
        if part_type is not None:
            payload["part_type"] = part_type
        payloads.append(payload)

        audit_rows.append({
            "segment_id": segment.id,
            "segment_index": segment.segment_index,
            "span_start": segment.span_start,
            "span_end": segment.span_end,
            "reviewed_title": segment_title,
            "reviewed_author": segment_author,
            "label": segment.label.name if segment.label else None,
            "part_type": part_type,
            "skipped_no_label": segment.label is None,
        })
    return payloads, audit_rows

## 5. Push one document's segments to BDRC

Same URL/method/headers as `bdrc.volume.update_volume`, but posts a hand-built dict instead of a `VolumeInput`/`SegmentInput` pair so `part_type="toc"` isn't stripped by that model's current `Literal["text", "editorial"]`. `status` is whatever `get_volume` reports for this volume right now, so this push doesn't change where the volume sits in the review workflow.

In [ ]:
async def push_volume_segments(volume_id: str, volume: Dict[str, Any], base_text: str, segment_payloads: list) -> Dict[str, Any]:
    url = f"{BDRC_BACKEND_URL}/volumes/{volume_id}"
    headers = {"accept": APPLICATION_JSON, "Content-Type": APPLICATION_JSON}
    payload = {
        "rep_id": volume.get("rep_id"),
        "vol_id": volume.get("vol_id"),
        "vol_version": volume.get("vol_version"),
        "status": volume.get("status"),
        "base_text": base_text,
        "segments": segment_payloads,
    }
    payload = {k: v for k, v in payload.items() if v is not None}

    client = await get_http_client()
    response = await client.post(url, json=payload, headers=headers)
    response.raise_for_status()
    return response.json()

## 6. Main loop

In [ ]:
async def run() -> Dict[str, Any]:
    db = SessionLocal()
    stats = {
        "volumes_seen": 0,
        "volumes_without_document": 0,
        "documents_pushed": 0,
        "segments_with_part_type": 0,
        "segments_unlabeled_skipped": 0,
        "errors": [],
    }
    audit_log: List[Dict[str, Any]] = []
    seen_volume_ids = set()
    try:
        async for volume_item in iter_all_volumes():
            volume_id = volume_item.get("id")
            if not volume_id or volume_id in seen_volume_ids:
                # Same volume can show up more than once if STATUSES overlap in practice.
                continue
            seen_volume_ids.add(volume_id)
            stats["volumes_seen"] += 1
            try:
                document = outliner_repo.fetch_document_by_filename(db, volume_id)
                if document is None:
                    stats["volumes_without_document"] += 1
                    continue
                if not document.segments:
                    continue

                # Full volume detail (rep_id/vol_id/vol_version/mw_id/status), same call
                # _push_document_segments_to_bdrc makes before building segment inputs.
                volume = await get_volume(volume_id)

                segment_payloads, audit_rows = build_segment_payloads(document, volume)
                for row in audit_rows:
                    audit_log.append({"volume_id": volume_id, "document_id": document.id, **row})
                    if row["skipped_no_label"]:
                        stats["segments_unlabeled_skipped"] += 1
                    else:
                        stats["segments_with_part_type"] += 1

                stats["documents_pushed"] += 1
                if not DRY_RUN:
                    await push_volume_segments(volume_id, volume, document.content, segment_payloads)
            except Exception as e:
                stats["errors"].append({"volume_id": volume_id, "detail": str(e)})
    finally:
        db.close()
        await aclose_http_client()
    return {"stats": stats, "audit_log": audit_log}

## 7. Execute

Leave `DRY_RUN = True` (set above) for the first pass and inspect `result["stats"]` / `result["audit_log"]` — no HTTP POSTs go out to BDRC in that mode. Only set `DRY_RUN = False` and rerun once the dry-run output looks right.

**Not run yet — run manually when ready.**

In [ ]:
result = await run()
result["stats"]

In [ ]:
# Inspect a sample of planned segment payloads before flipping DRY_RUN off.
import json
print(json.dumps(result["audit_log"][:20], indent=2, default=str))